# Loader

## LLM 
> 이미지가 있는 pdf 파일을 해석하는 용도로 사용 

In [1]:
from langchain_openai import ChatOpenAI 

llm = ChatOpenAI(
    model="gpt-6-luna"
)

## PDF Loader

In [2]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader 
from langchain_community.document_loaders.parsers import LLMImageBlobParser

pdf_file_path = "./data/LG Aimers 4기 소개자료.pdf"
loader = PyMuPDF4LLMLoader(
    file_path=pdf_file_path,
    mode='page',
    extract_images=True,
    images_parser=LLMImageBlobParser(
        model=llm
    )
)

In [3]:
docs = loader.load()
print(f"pdf 파일 로드 수: {len(docs)}")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
pdf 파일 로드 수: 7


In [9]:
docs[2].metadata

{'producer': '',
 'creator': '',
 'creationdate': '2025-07-15T07:40:32+00:00',
 'source': './data/LG Aimers 4기 소개자료.pdf',
 'file_path': './data/LG Aimers 4기 소개자료.pdf',
 'total_pages': 7,
 'format': 'PDF 1.7',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2025-07-15T07:40:32+00:00',
 'trapped': '',
 'modDate': "D:20250715074032+00'00'",
 'creationDate': "D:20250715074032+00'00'",
 'page': 2}

In [8]:
print(docs[2].page_content)

### **1. LG Aimers 프로그램개요**



![- **요약:** 한국어 교육 콘텐츠 1달 과정 구성도. 온라인 초급 콘텐츠(외부), 온라인 중·상급 콘텐츠(내부), LG AI연구위원 온라인 특강을 더하고, 무료 콘텐츠 발굴·재구성과 중급 콘텐츠 10모듈 개선(30개 강좌, 대학원 전공 1개 과목 수준)을 제시함.
- **이미지 텍스트:**
  - 교육 콘텐츠 (1달 과정)
  - 온라인 초급 콘텐츠 (외부)
  - 온라인 중/상급 콘텐츠 (내부)
  - LG AI연구위원 온라인 특강
  - 무료 콘텐츠 발굴 / 재구성
  - 중급 콘텐츠 10모듈 개선 (30개 강좌)
  - : 대학원 전공 1개 과목 수준](#)





![- **이미지 요약:** 회색 배너에 초록색 원형 번호 1과 외부 교육생이 이용할 수 있는 교육 플랫폼 안내 문구가 표시되어 있음.
- **이미지 텍스트:** ① 외부 교육생이 수강할 수 있는 교육 플랫폼](#)



![**이미지 요약:** 초록색 원형 아이콘에 숫자 2가 있고, 해커톤 운영 플랫폼을 소개하는 한국어 제목이 적힌 배너.

**이미지 텍스트:**  
2  
해커톤 운영(리더보드, 랭킹, 커뮤니티)을 위한 플랫폼](#)















LG AI연구원 자체 LMS 플랫폼 개발


 - LG Aimers 전용플랫폼


해커톤 운영 스타트업과 협업


 - 해커톤노하우가있는 DACON 과 협업으로진행


LG AI연구원 직접 제작 (국내 top tier 교수님 참여)


- AI 연구위원, AI 전공 교수진이참여하여 대학원과정

수준의 AI 교육프로그램을 제작


매 기수별해커톤문제에따라교육커리큘럼제작



![**요약:** 회색 테두리의 안내 배너. 초록색 원 안에 숫자 3이 있고, ‘온라인중 / 상급 콘텐츠 자체 제작’이라고 적혀 있다.

**텍스트:**  
3  
온라인중 / 상급 콘텐츠 자체 제작](#)



![## 이미지 요약
온라인 예선과 오프라인 본선으로 진행되는 해커톤 일정 및 참가 규모,

# Split

In [ ]:
# page별 문서의 길이가 너무 길기 때문에 새로운 Section을 생성해서 Split 진행
max(len(doc.page_content) for doc in docs)

3977

## 의미론적으로 문서들의 Section 생성하기 

### OutputParser

In [14]:
from pydantic import BaseModel, Field 

class Section(BaseModel):
    title: str = Field(description="문서의 내용을 바탕으로 목차 이름 생성")
    start_text: str = Field(description="해당 목차가 실행되는 문자")

class ListSection(BaseModel):
    sections: list[Section]

In [15]:
from langchain_core.output_parsers import PydanticOutputParser

output_parser = PydanticOutputParser(pydantic_object=ListSection)

In [16]:
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Section": {"properties": {"title": {"description": "문서의 내용을 바탕으로 목차 이름 생성", "title": "Title", "type": "string"}, "start_text": {"description": "해당 목차가 실행되는 문자", "title": "Start Text", "type": "string"}}, "required": ["title", "start_text"], "title": "Section", "type": "object"}}, "properties": {"sections": {"items": {"$ref": "#/$defs/Section"}, "title": "Sections", "type": "array"}}, "required": ["sections"]}
```


### Prompt

In [17]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""당신은 PDF 문서의 구조를 분석하는 전문 AI입니다.

주어진 문서의 내용을 분석하여 의미적으로 구분되는 목차(Section)를 생성하고, 각 목차의 제목과 시작 위치를 추출하세요.

## 작업 지침

1. 문서 전체의 내용과 흐름을 분석하세요.
2. 문서에 명시적인 목차가 있다면 이를 우선 참고하세요.
3. 명시적인 목차가 없거나 구조가 불명확하다면 내용의 의미와 주제 전환을 기준으로 목차를 생성하세요.
4. 각 목차에는 해당 내용을 대표하는 명확하고 간결한 제목을 부여하세요.
5. 각 목차가 시작되는 위치를 원본 문서에서 정확히 찾아 추출하세요.
6. 시작 위치는 해당 Section의 첫 부분에 등장하는 원문 문자열(`start_text`)로 표현하세요.
7. `start_text`는 원본 문서에 실제 존재하는 문자열을 변경하지 않고 그대로 추출해야 합니다.
8. 각 Section의 시작 문자열은 서로 겹치지 않아야 합니다.
9. 목차는 원본 문서에 등장하는 순서를 유지하세요.
10. 임의의 내용이나 존재하지 않는 위치를 생성하지 마세요.

## Section 분리 기준

- 주제나 설명 대상이 명확하게 변경되는 지점
- 새로운 개념, 방법론 또는 실험이 시작되는 지점
- 논리적으로 독립적인 내용을 설명하는 지점
- 문서에 존재하는 장, 절, 소제목 등의 구조적 경계

단순한 문단 변경이나 페이지 변경만으로 새로운 Section을 생성하지 마세요.

## 출력 형식
{format_instructions}

## 분석할 문서
{full_text}
""",
    input_variables=["full_text"],
    partial_variables={"format_instructions":format_instructions}
)

### Model

In [18]:
from langchain_openai import ChatOpenAI 

model = ChatOpenAI(
    model="gpt-6-luna"
)

### Chain

In [19]:
chain = prompt | model | output_parser 

In [20]:
full_text = "\n\n".join(
    doc.page_content for doc in docs
)

len(full_text)

9817

In [21]:
print(full_text[:100])

# **LG Aimers** **소개 자료**


###### LG AI연구원｜



![**Image summary:** Minimal graphic with a thick ma


In [22]:
result = chain.invoke({
    "full_text":full_text
})

In [23]:
# LLM이 만들 전체 목차의 수 
len(result.sections)

8

In [24]:
result.sections[1].title, result.sections[1].start_text

('교육 콘텐츠 및 해커톤 운영 체계', '### **1. LG Aimers 프로그램개요**')

In [25]:
start_index = full_text.find(result.sections[1].start_text)
end_index = full_text.find(result.sections[2].start_text)

start_index, end_index

(513, 2860)

In [27]:
print(full_text[start_index:end_index])

### **1. LG Aimers 프로그램개요**

###### ❑ 교육과 경험의 기회를 필요로 하는 청년들에게 양질의 AI교육을 온라인으로 제공하고, 기업의 실제 data를 다루며 실무를 경험할 수 있는 기회를 제공하기 위해 『온라인 AI전문가 과정(1개월)』과『AI 해커톤 (Hackathon) (1개월)』을 **결합한 형태의 교육 프로그램** **-** 온라인 AI 교육과정: 국내 AI 전문가 (현업 전문가, 저명 교수) 의 최신 AI 기법에관한 온라인 강의(1개월)


     **온라인 초급 교육은 기초적인 프로그래밍데이터 처리에 관련된 내용들로써 외부 교육 콘텐츠를 활용(optional)**


          **초급 교육은 자료구조와 알고리즘 등 컴퓨팅 관련 기본 소양 교육**


     **온라인 중상급 교육은 국내 AI 분야전문가들이 주제별로 강의 (LG 자체 개발-** **부록 참조)**


          **중상급 교육은 대학원 수준의 AI 요소 기술 및 이론에 관한 교육으로 이루어져 있음**


          **중상급 교육을 받기 전에 학부 수준의 ‘인공지능 개론’ 과목을 이수하는 것을 추천**

###### **-** **온라인 AI 해커톤: LG계열사의문제를 현장의 실제data를활용하여 해결하는 해커톤 예선(1개월)** **-** **오프라인 AI 해커톤: 온라인 해커톤에서 선발된 본선 진출자들만 참가 (1박2일)** ❑ 수료 조건 : 온라인 AI전문가 과정이수(교육영상100% 이수및퀴즈참여) & AI 해커톤 baseline model 성능이상 **수료자에 한하여 수료증 발급** ❑ 대상 및 규모 : 만 19세에서 29세의 청년 (미취업자 대상) ❑ 운영 일정 : 매년 연 2회 진행. 여름방학(7~8월), 겨울방학(1~2월)


❑ **LG Aimers 채널:** **[https://lgaimers.ai/](https://lgaimers.ai/)**




### **1. LG Aimers 프로그램개요**



![- **

## 생성된 Section을 기반으로 Documents 생성하기 

### section별 시작 index 생성

In [28]:
section_postions = []
for section in result.sections:
    # 시작 문자열의 위치정보 찾기 
    start_index = full_text.find(section.start_text)

    if start_index == -1:
        raise ValueError(f"해당 시작 문자의 위치정보를 찾지 못했습니다 > {section.start_text}")

    section_postions.append({
        "title": section.title,
        "start_index": start_index
    })

# 시작 위치정보를 기준으로 정렬
section_postions.sort(key=lambda x: x['start_index'])

### Document List 생성하기 

In [29]:
from langchain_core.documents import Document 

docs_by_section = []
for i, section in enumerate(section_postions):
    start_index = section['start_index']

    if i + 1 < len(section_postions):
        end_index = section_postions[i+1]['start_index']
    else:
        end_index = len(full_text)

    docs_by_section.append(
        Document(
            page_content=full_text[start_index:end_index].strip(),
            metadata={
                "section": section['title'],
                "source": pdf_file_path
            }
        )
    )

In [30]:
# 검증
# 새로 만들어진 리스트의 길이와 LLM이 만든 목차의 길이가 같으면 성공!!
len(docs_by_section) == len(result.sections)

True

In [31]:
print(f"제목: {docs_by_section[1].metadata['section']}")
print("="*50)
print(docs_by_section[1].page_content)

제목: 교육 콘텐츠 및 해커톤 운영 체계
### **1. LG Aimers 프로그램개요**

###### ❑ 교육과 경험의 기회를 필요로 하는 청년들에게 양질의 AI교육을 온라인으로 제공하고, 기업의 실제 data를 다루며 실무를 경험할 수 있는 기회를 제공하기 위해 『온라인 AI전문가 과정(1개월)』과『AI 해커톤 (Hackathon) (1개월)』을 **결합한 형태의 교육 프로그램** **-** 온라인 AI 교육과정: 국내 AI 전문가 (현업 전문가, 저명 교수) 의 최신 AI 기법에관한 온라인 강의(1개월)


     **온라인 초급 교육은 기초적인 프로그래밍데이터 처리에 관련된 내용들로써 외부 교육 콘텐츠를 활용(optional)**


          **초급 교육은 자료구조와 알고리즘 등 컴퓨팅 관련 기본 소양 교육**


     **온라인 중상급 교육은 국내 AI 분야전문가들이 주제별로 강의 (LG 자체 개발-** **부록 참조)**


          **중상급 교육은 대학원 수준의 AI 요소 기술 및 이론에 관한 교육으로 이루어져 있음**


          **중상급 교육을 받기 전에 학부 수준의 ‘인공지능 개론’ 과목을 이수하는 것을 추천**

###### **-** **온라인 AI 해커톤: LG계열사의문제를 현장의 실제data를활용하여 해결하는 해커톤 예선(1개월)** **-** **오프라인 AI 해커톤: 온라인 해커톤에서 선발된 본선 진출자들만 참가 (1박2일)** ❑ 수료 조건 : 온라인 AI전문가 과정이수(교육영상100% 이수및퀴즈참여) & AI 해커톤 baseline model 성능이상 **수료자에 한하여 수료증 발급** ❑ 대상 및 규모 : 만 19세에서 29세의 청년 (미취업자 대상) ❑ 운영 일정 : 매년 연 2회 진행. 여름방학(7~8월), 겨울방학(1~2월)


❑ **LG Aimers 채널:** **[https://lgaimers.ai/](https://lgaimers.ai/)**




### **1. LG Ai

## 규칙 기반으로 문서를 분리하자

In [ ]:
# 문자의 길이가 아직도 너무 길기 때문에 이번에는 규칙 기반으로 분리하자
max(len(doc.page_content) for doc in docs_by_section)

2583

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 

splitter = RecursiveCharacterTextSplitter(
    # 청크 크기를 매우 작게 설정합니다. 예시를 위한 설정입니다.
    chunk_size=500,
    # 청크 간의 중복되는 문자 수를 설정합니다.
    chunk_overlap=50,
    # 문자열 길이를 계산하는 함수를 지정합니다.
    length_function=len,
    # 구분자로 정규식을 사용할지 여부를 설정합니다.
    is_separator_regex=False,
    # (옵션)한국어 문장 구분용 구분자 설정
    separators=[
        "\n\n",  # 문단
        "\n",    # 줄바꿈
        r"(?<=[.?!])\s+",  # 영어 문장
        r"(?<=[다요죠음함임니다])[\.\?!]?\s+",  # 한국어 종결 표현
        " ",      # 단어
        "",       # 최후의 수단: 글자 단위
    ]
)

In [34]:
docs_by_splitter = splitter.split_documents(docs_by_section)

In [36]:
print(f"의미론적으로 분리한 숫자: {len(docs_by_section)}")
print("="*50)
print(f"의미 & 규칙 기반으로 분리한 숫자: {len(docs_by_splitter)}")

의미론적으로 분리한 숫자: 8
의미 & 규칙 기반으로 분리한 숫자: 25


# Embedding Model

In [37]:
from langchain_ollama import OllamaEmbeddings 

embedding_model = OllamaEmbeddings(
    model="qwen3-embedding:0.6b"
)

In [38]:
msg = "안녕하세요?"
vector_msg = embedding_model.embed_query(msg)
len(vector_msg)

1024

# VectorStore

## VectorDB 접속 확인 

In [39]:
from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333)
print("접속 성공")

접속 성공


## Upload Documents

In [40]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=docs_by_splitter,
    embedding=embedding_model,
    collection_name="advanced_rag",
    location="http://localhost:6333"
)

## Test

In [41]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

In [42]:
search_docs = retriever.invoke("온라인 해커톤의 일정을 설명해줘.")

In [43]:
print(search_docs[0].page_content)

**텍스트:**  
3  
온라인중 / 상급 콘텐츠 자체 제작](#)



![## 이미지 요약
온라인 예선과 오프라인 본선으로 진행되는 해커톤 일정 및 참가 규모, 채용 연계 내용을 나타낸 흐름도. 예선은 총 4주간 진행되며, 본선에서는 합숙 경연·최종 평가·시상식과 채용 박람회가 운영됨.

## 이미지 내 텍스트
해커톤

온라인 해커톤 (예선)  
과제부여  
(Data Set)  
대회 진행  
(리더보드 운영)  
예선 종료 및  
1차심사  
(총 4주간 진행)

오프라인 해커톤 (본선)  
합숙경연  
(1박 2일)  
최종평가  
(상위팀 PT)  
시상식  
(상장, 상금)  
채용 박람회 동시 운영

예상 참석인원 약 2,000명  
본선 참가 인원 약 100명 (30개팀)  
(인재pool등록 및 각 사 채용면담 진행)  
1~3위 입상자 선정  
(즉시 면접 및 채용)](#)
